# Fire Emergency Call Data – Cleaning & Feature Extraction for ML

**Source:** `UPDATED DETAILS OF FIRE EMERGENCY CALL.xlsx` (one sheet per month + `SUMMARY OF FIRE`)

**What this notebook does**
1. Loads every monthly sheet automatically (new months are picked up without code changes)
2. Cleans dates (fixes DD/MM vs MM/DD swaps using the sheet month), times, and locations
3. Calculates response time in minutes
4. Extracts ML features from the free-text columns (cause, fuel, equipment, agent used, extinguisher count/kg, injuries, who put out the fire, etc.)
5. Builds recurrence features (repeat equipment, days since last fire at the same unit)
6. Creates three ML-ready datasets:
   - **Incident-level** (classification: fire nature, cause, severity)
   - **Daily × Area panel** (prediction: *will a fire happen at this area on this day?*)
   - **Daily plant-level** (time-series: fire count per day)
7. Validates against the `SUMMARY OF FIRE` sheet
8. **Exports** everything to Excel + CSV

## 1. Setup

In [ ]:
import re
import warnings
from pathlib import Path
from datetime import datetime, time

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# ---- CONFIG: change paths if needed ----
INPUT_FILE = Path("UPDATED DETAILS OF FIRE EMERGENCY CALL.xlsx")
OUTPUT_DIR = Path("fire_ml_output")
YEAR = 2026

MONTHS = {m: i for i, m in enumerate(
    ["JANUARY", "FEBRUARY", "MARCH", "APRIL", "MAY", "JUNE",
     "JULY", "AUGUST", "SEPTEMBER", "OCTOBER", "NOVEMBER", "DECEMBER"], start=1)}

## 2. Load all monthly sheets
The header row is detected by searching for the `DATE` cell, so title rows / blank rows are skipped automatically.

In [ ]:
xls = pd.ExcelFile(INPUT_FILE)
print("Sheets:", xls.sheet_names)

def load_month_sheet(sheet_name):
    raw = pd.read_excel(INPUT_FILE, sheet_name=sheet_name, header=None, dtype=object)
    hdr_idx = raw.index[raw.iloc[:, 0].astype(str).str.strip().str.upper() == "DATE"]
    if len(hdr_idx) == 0:
        return None
    h = hdr_idx[0]
    df = raw.iloc[h + 1:].copy()
    df.columns = [str(c).strip() for c in raw.iloc[h]]
    df = df.dropna(how="all")
    df["SHEET_MONTH"] = sheet_name.strip().upper()
    df["SHEET_MONTH_NUM"] = MONTHS[sheet_name.strip().upper()]
    df["SOURCE_ROW"] = df.index + 1  # Excel row number for traceability
    return df

frames = [load_month_sheet(s) for s in xls.sheet_names if s.strip().upper() in MONTHS]
raw_df = pd.concat([f for f in frames if f is not None], ignore_index=True)

raw_df = raw_df.rename(columns={
    "DATE": "date_raw", "TIME": "call_time_raw", "LOCATION": "location_raw",
    "EMERGENCY DETAILS": "emergency_details", "RESPONSE TIME": "response_time_raw",
    "ACTION TAKEN": "action_taken", "Compliance / Closure": "closure_notes",
})
print("Records loaded:", len(raw_df))
raw_df.groupby("SHEET_MONTH", sort=False).size()

Sheets: ['JANUARY', 'FEBRUARY ', 'MARCH', 'APRIL', 'MAY', 'JUNE', 'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'SUMMARY OF FIRE']
Records loaded: 61


SHEET_MONTH
JANUARY       7
FEBRUARY     11
MARCH         2
APRIL         5
MAY           4
JUNE         12
JULY         11
AUGUST        5
SEPTEMBER     3
OCTOBER       1
dtype: int64

## 3. Basic text cleaning

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\u2011", "-").replace("\u2010", "-").replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s if s not in ("", "-", "\\-") else np.nan

df = raw_df.copy()
for c in ["call_time_raw", "location_raw", "emergency_details", "response_time_raw",
          "action_taken", "closure_notes"]:
    df[c] = df[c].apply(clean_text)

# drop rows with no date AND no details (pure blanks)
df = df[~(df["date_raw"].isna() & df["emergency_details"].isna())].reset_index(drop=True)
df[["date_raw", "call_time_raw", "location_raw", "response_time_raw"]].head(10)

,date_raw,call_time_raw,location_raw,response_time_raw
0,2026-01-01 00:00:00,2116HRS,GANTRY 3 BAY 21,2117HRS
1,2026-06-01 00:00:00,0320HRS,RFCC,0322HRS
2,13/1/2026,1400HRS,MSB Battery limit,1402HRS
3,22/1/2026,1839HRS,MSB CCR UNIT 153,1840HRS
4,24/1/2026,2240HRS,CDU CT-2,2241HRS
5,25/1/2026,1921HRS,MHC,1922HRS
6,31/1/2026,1026HRS,MSB,1027HRS
7,2026-02-02 00:00:00,1132HRS,OFFSHORE,1133HRS
8,2026-02-02 00:00:00,1452HRS,CDU Compressor side,1454HRS
9,2026-02-04 00:00:00,0635HRS,MHC,0636HRS


## 4. Fix dates
Excel stored some dates as real dates and others as text (`13/1/2026`). Some real dates were mis-read as MM/DD
(e.g. **06/01/2026 in the JANUARY sheet became 1-Jun**). Rule: **the sheet month is the truth** – if the parsed
month ≠ sheet month but the day = sheet month, day and month are swapped.

In [ ]:
def fix_date(val, sheet_month):
    if pd.isna(val):
        return pd.NaT, "missing"
    if isinstance(val, (datetime, pd.Timestamp)):
        d, m, y = val.day, val.month, val.year
    else:
        s = str(val).strip()
        parts = re.split(r"[/\-.]", s)
        if len(parts) != 3:
            return pd.NaT, "unparsed"
        a, b, y = int(parts[0]), int(parts[1]), int(parts[2])
        # decide which part is the month using the sheet month
        if b == sheet_month:
            d, m = a, b
        elif a == sheet_month:
            d, m = b, a
        else:
            d, m = a, b
    if m == sheet_month:
        return pd.Timestamp(y, m, d), "ok"
    if d == sheet_month and m <= 31:
        return pd.Timestamp(y, d, m), "day_month_swapped"
    return pd.Timestamp(y, m, d), "month_mismatch_check"

res = df.apply(lambda r: fix_date(r["date_raw"], r["SHEET_MONTH_NUM"]), axis=1)
df["incident_date"] = [r[0] for r in res]
df["date_fix_flag"] = [r[1] for r in res]
print(df["date_fix_flag"].value_counts())
df.loc[df["date_fix_flag"] != "ok", ["SHEET_MONTH", "date_raw", "incident_date", "date_fix_flag"]]

date_fix_flag
ok                   60
day_month_swapped     1
Name: count, dtype: int64


,SHEET_MONTH,date_raw,incident_date,date_fix_flag
1,JANUARY,2026-06-01 00:00:00,2026-01-06,day_month_swapped


## 5. Fix times & calculate response time
Handles `2116HRS`, `938HRS`, `000HRS`, numeric `1550`, and blanks. Midnight roll-over is handled.

In [ ]:
def parse_hhmm(val):
    if pd.isna(val):
        return None
    digits = re.sub(r"[^0-9]", "", str(val))
    if digits == "":
        return None
    digits = digits.zfill(4)[-4:]
    hh, mm = int(digits[:2]), int(digits[2:])
    if hh > 23 or mm > 59:
        return None
    return time(hh, mm)

df["call_time"] = df["call_time_raw"].apply(parse_hhmm)
df["response_time"] = df["response_time_raw"].apply(parse_hhmm)

def combine(d, t):
    return pd.Timestamp.combine(d.date(), t) if (pd.notna(d) and t is not None) else pd.NaT

df["call_datetime"] = [combine(d, t) for d, t in zip(df["incident_date"], df["call_time"])]
df["response_datetime"] = [combine(d, t) for d, t in zip(df["incident_date"], df["response_time"])]
# roll-over past midnight
mask = df["response_datetime"] < df["call_datetime"]
df.loc[mask, "response_datetime"] += pd.Timedelta(days=1)

df["response_minutes"] = (df["response_datetime"] - df["call_datetime"]).dt.total_seconds() / 60
df["time_missing_flag"] = df["call_time"].isna().astype(int)

df[["incident_date", "call_time_raw", "call_time", "response_time_raw", "response_minutes"]].head(10)

,incident_date,call_time_raw,call_time,response_time_raw,response_minutes
0,2026-01-01,2116HRS,21:16:00,2117HRS,1.0
1,2026-01-06,0320HRS,03:20:00,0322HRS,2.0
2,2026-01-13,1400HRS,14:00:00,1402HRS,2.0
3,2026-01-22,1839HRS,18:39:00,1840HRS,1.0
4,2026-01-24,2240HRS,22:40:00,2241HRS,1.0
5,2026-01-25,1921HRS,19:21:00,1922HRS,1.0
6,2026-01-31,1026HRS,10:26:00,1027HRS,1.0
7,2026-02-02,1132HRS,11:32:00,1133HRS,1.0
8,2026-02-02,1452HRS,14:52:00,1454HRS,2.0
9,2026-02-04,0635HRS,06:35:00,0636HRS,1.0


## 6. Time-based features

In [ ]:
df["month"] = df["incident_date"].dt.month
df["month_name"] = df["incident_date"].dt.month_name()
df["quarter"] = df["incident_date"].dt.quarter
df["week_of_year"] = df["incident_date"].dt.isocalendar().week.astype(int)
df["day_of_month"] = df["incident_date"].dt.day
df["day_of_week"] = df["incident_date"].dt.dayofweek          # 0=Mon
df["day_name"] = df["incident_date"].dt.day_name()
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["hour"] = df["call_datetime"].dt.hour

def shift_of(h):
    if pd.isna(h): return "Unknown"
    return "Day (07-19)" if 7 <= h < 19 else "Night (19-07)"

def time_band(h):
    if pd.isna(h): return "Unknown"
    h = int(h)
    if 0 <= h < 6: return "Early Morning (00-06)"
    if 6 <= h < 12: return "Morning (06-12)"
    if 12 <= h < 18: return "Afternoon (12-18)"
    return "Evening (18-24)"

df["shift"] = df["hour"].apply(shift_of)
df["time_band"] = df["hour"].apply(time_band)
# Nigerian seasons: rainy ~Apr–Oct, dry ~Nov–Mar (lightning risk in rainy season)
df["season"] = np.where(df["month"].between(4, 10), "Rainy", "Dry")
# cyclical encodings for ML
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df[["incident_date", "hour", "shift", "time_band", "day_name", "season"]].head()

,incident_date,hour,shift,time_band,day_name,season
0,2026-01-01,21.0,Night (19-07),Evening (18-24),Thursday,Dry
1,2026-01-06,3.0,Night (19-07),Early Morning (00-06),Tuesday,Dry
2,2026-01-13,14.0,Day (07-19),Afternoon (12-18),Tuesday,Dry
3,2026-01-22,18.0,Day (07-19),Evening (18-24),Thursday,Dry
4,2026-01-24,22.0,Night (19-07),Evening (18-24),Saturday,Dry


## 7. Standardise location → Area & Zone
Rules are ordered (first match wins). Add new keywords to `AREA_RULES` as new locations appear.

In [ ]:
AREA_RULES = [  # (regex on location + details, AREA, ZONE)
    (r"\bSS-?\d+",                                 "Substation",        "Utilities & Power"),
    (r"OFFSHORE",                                  "Offshore",          "Offshore"),
    (r"\bMHC\b",                                   "MHC",               "Process Unit"),
    (r"\bRFCCU?\b",                                "RFCC",              "Process Unit"),
    (r"\bMSB\b|MS BLOCK|CCR",                      "MSB",               "Process Unit"),
    (r"\bCDU\b",                                   "CDU",               "Process Unit"),
    (r"\bHGU\b",                                   "HGU",               "Process Unit"),
    (r"\bPPU\b",                                   "PPU",               "Process Unit"),
    (r"\bITF\b",                                   "ITF (Tank Farm)",   "Tank Farm"),
    (r"SURGE TANK",                                "Surge Tank",        "Tank Farm"),
    (r"\bCPP\b",                                   "CPP",               "Utilities & Power"),
    (r"COOLING TOWER|\bCT-?\d",                    "Cooling Tower",     "Utilities & Power"),
    (r"CONTROL ROOM",                              "Control Room",      "Utilities & Power"),
    (r"GANTRY",                                    "Loading Gantry",    "Loading Gantry"),
    (r"CNG",                                       "CNG Dispenser",     "Loading Gantry"),
    (r"22K",                                       "22K Accommodation", "Accommodation"),
    (r"10K",                                       "10K Accommodation", "Accommodation"),
    (r"MAIN GATE",                                 "Main Gate",         "Gate & Roads"),
    (r"ADMIN",                                     "Admin Block",       "Admin"),
    (r"FERTILIZER",                                "Fertilizer Premises", "Third-Party Premises"),
]

def map_area(loc, details):
    loc_u = str(loc).upper() if pd.notna(loc) else ""
    det_u = str(details).upper() if pd.notna(details) else ""
    # 1st pass: location only; 2nd pass: details (for vague locations)
    for text in (loc_u, det_u):
        for pat, area, zone in AREA_RULES:
            if re.search(pat, text):
                return area, zone
    return "Other", "Other"

ar = df.apply(lambda r: map_area(r["location_raw"], r["emergency_details"]), axis=1)
df["area"] = [a for a, _ in ar]
df["zone"] = [z for _, z in ar]
df["location_clean"] = df["location_raw"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
df[["location_raw", "area", "zone"]].drop_duplicates().sort_values("zone")

,location_raw,area,zone
57,"Block 4, 10k Accomodation area",10K Accommodation,Accommodation
48,22k BLK A-60 & A-61,22K Accommodation,Accommodation
47,XCMG Kitchen 22k accomodation area,22K Accommodation,Accommodation
44,22K BLK M-03,22K Accommodation,Accommodation
37,RM-01 M-03 22K ACCOMODATION AREA,22K Accommodation,Accommodation
33,22K A78 RM-1,22K Accommodation,Accommodation
31,22K(M3 RM-5),22K Accommodation,Accommodation
30,22K(Block K-55 ),22K Accommodation,Accommodation
13,Admin blcok 21C,Admin Block,Admin
60,Refinery Main Gate,Main Gate,Gate & Roads


## 8. Feature extraction from EMERGENCY DETAILS (keyword / regex NLP)

In [ ]:
def first_match(text, rules, default="Unknown"):
    t = str(text).upper() if pd.notna(text) else ""
    for pat, label in rules:
        if re.search(pat, t):
            return label
    return default

def any_match(text, pat):
    t = str(text).upper() if pd.notna(text) else ""
    return int(bool(re.search(pat, t)))

CAUSE_RULES = [
    (r"LIGHTNING",                                            "Lightning strike"),
    (r"ARSON|MISCREA|SCAVENGER|RECOVERY OF COPPER",           "Arson / Vandalism / Scavenging"),
    (r"HOT JOB|HOT WORK|WELDING|GRINDING|CUTTING|ARC WELD|PUNCH|PUNCTURE|MODIFICATION JOB", "Hot work / Maintenance activity"),
    (r"PYROPHORIC",                                           "Pyrophoric"),
    (r"KETTLE|COOK|BURNER|KITCHEN|GAS COOKER",                "Cooking / Domestic appliance"),
    (r"VEHICLE|CAR\b|TRUCK|TANKER|HILUX|CATERPILLAR|ENGINE COMPARTMENT|CHASSIS", "Vehicle fault"),
    (r"ELECTRIC|PANEL|CABLE|POWER SURGE|OVERLOAD|SPARK|VFD|BLOWER", "Electrical fault"),
    (r"FLANGE|LEAK|SEAL",                                     "Flange / Seal leak"),
    (r"SOAKED INSULATION|LAGGING|HOT SURFACE|HOT PIPELINE|OIL DRIPPING", "Hot surface / Soaked insulation"),
    (r"OVERHEAT|HIGH TEMPERATURE|RISE IN TEMPERATURE",        "Overheating"),
    (r"VENT",                                                 "Vent release"),
    (r"DUMP",                                                 "Dumped material"),
    (r"BLAST|EXPLO",                                          "Equipment failure / Blast"),
]

FUEL_RULES = [
    (r"HYDROGEN|LPD",                                         "Hydrogen"),
    (r"NATURAL GAS|\bCNG\b",                                  "Natural gas"),
    (r"DIESEL|IFO",                                           "Diesel / Fuel oil"),
    (r"PMS|GASOLINE|NAPHTHA|ISOMERATE",                       "Gasoline / Naphtha"),
    (r"HYDROCARBON|EFFLUENT|\bOIL\b|REACTOR|RECYCLE GAS|FEED",    "Hydrocarbon (process)"),
    (r"ELECTRIC|PANEL|CABLE|POWER|VFD|BLOWER",                "Electrical"),
    (r"VEHICLE|CAR\b|TRUCK|TANKER|HILUX|CATERPILLAR",         "Vehicle materials"),
    (r"KETTLE|POT|COOK",                                      "Domestic / Cooking"),
    (r"INSULATION",                                           "Insulation material"),
]

EQUIP_RULES = [
    (r"VEHICLE|\bCAR\b|TRUCK|TANKER|HILUX|CATERPILLAR",        "Vehicle"),
    (r"KETTLE|COOKER|BURNER|\bPOT\b",                        "Domestic appliance"),
    (r"EXCHANGER|\bEE-?\d|-EE-|\bEE\b|CONDENSER|PRE-HTR|HEATER",  "Heat exchanger / Heater"),
    (r"REACTOR|\bRB-",                                       "Reactor"),
    (r"PUMP|-PA-?\d",                                        "Pump"),
    (r"COMPRESSOR|-KA-",                                     "Compressor"),
    (r"TURBINE|EXPANDER",                                    "Turbine / Expander"),
    (r"\bTANKS?\b|-TT-",                                     "Storage tank"),
    (r"PANEL",                                               "Electrical panel"),
    (r"CABLE",                                               "Cable"),
    (r"LOADING ARM|GANTRY",                                  "Loading arm"),
    (r"MOV|VALVE|STRAINER|DRAIN PIPE|LINE|HEADER|VENT",      "Piping / Valve / Line"),
    (r"ROOM|TOILET|BLOCK|APARTMENT",                         "Building structure"),
]

ACTIVITY_RULES = [
    (r"START.?UP|WARM.?UP|BROUGHT ONLINE",                   "Start-up"),
    (r"SHUTDOWN|SHUT DOWN",                                  "Shutdown"),
    (r"HOT JOB|HOT WORK|WELDING|GRINDING|CUTTING|MODIFICATION|MAINTENANCE|DE-BLINDING|REMOVING|ERRECTION|ERECTION|CLEANING|INSULATION WORKER", "Maintenance / Project work"),
    (r"LOADING|DECANT",                                      "Loading / Unloading"),
    (r"FEED CHANGE",                                         "Process change"),
]

txt = df["emergency_details"].fillna("") + " " + df["location_raw"].fillna("")
df["cause_category"] = txt.apply(lambda t: first_match(t, CAUSE_RULES, "Other / Unknown"))
df["fuel_type"] = txt.apply(lambda t: first_match(t, FUEL_RULES, "Unknown"))
df["operating_activity"] = txt.apply(lambda t: first_match(t, ACTIVITY_RULES, "Normal operation"))

# equipment tag(s) e.g. 121-EE-1004, 152-RB-1001, 716-TT-1601A
TAG_RE = r"\b\d{3}\s?-?\s?[A-Z]{2}\s?-?\s?\d{4,5}[A-Z]?\b"
def extract_tags(text):
    if pd.isna(text): return np.nan
    tags = re.findall(TAG_RE, str(text).upper().replace("‑", "-"))
    tags = [re.sub(r"\s", "", t) for t in tags]
    tags = [re.sub(r"^(\d{3})-?([A-Z]{2})-?(\d+[A-Z]?)$", r"\1-\2-\3", t) for t in tags]
    return ", ".join(dict.fromkeys(tags)) if tags else np.nan

df["equipment_tags"] = df["emergency_details"].apply(extract_tags)
df["primary_tag"] = df["equipment_tags"].str.split(",").str[0].str.strip()
df["primary_tag_base"] = df["primary_tag"].str.replace(r"[A-Z]$", "", regex=True)  # 121-EE-1005A -> 121-EE-1005
df["unit_number"] = df["primary_tag"].str[:3]

# equipment type: use ISA-style tag code first (most reliable), else keyword rules
TAG_CODE_MAP = {"EE": "Heat exchanger / Heater", "EU": "Heat exchanger / Heater", "PA": "Pump",
                "RB": "Reactor", "KA": "Compressor", "TT": "Storage tank", "VV": "Vessel / Drum",
                "BL": "Blower / Fan", "FM": "Turbine / Expander"}
tag_equip = df["primary_tag"].str[4:6].map(TAG_CODE_MAP)
df["equipment_type"] = tag_equip.fillna(txt.apply(lambda t: first_match(t, EQUIP_RULES, "Unknown")))

# binary flags
D = df["emergency_details"]
df["is_flash_fire"]      = D.apply(lambda t: any_match(t, r"FLASH FIRE"))
df["is_smoke_only"]      = D.apply(lambda t: any_match(t, r"SMOKE") & (1 - any_match(t, r"FIRE OUTBREAK|OPEN FIRE")))
df["is_explosion_blast"] = D.apply(lambda t: any_match(t, r"BLAST|EXPLO"))
df["is_minor"]           = D.apply(lambda t: any_match(t, r"\bMINOR\b|SLIGHT"))
df["fire_spread"]        = D.apply(lambda t: any_match(t, r"SPREAD|ESCALAT|ENGULF"))
df["hydrocarbon_involved"] = D.apply(lambda t: any_match(t, r"HYDROCARBON|DIESEL|\bOIL\b|NAPHTHA|GASOLINE|PMS|IFO|ISOMERATE|EFFLUENT|NATURAL GAS"))
df["hydrogen_involved"]  = D.apply(lambda t: any_match(t, r"HYDROGEN"))
df["contractor_involved"] = (df["emergency_details"].fillna("") + " " + df["action_taken"].fillna("")).apply(
    lambda t: any_match(t, r"CONTRACTOR|M/S|KOINS|ENERGIO|POWER MECH|HR TECH|SINOMA|XCMG"))
df["detected_by_system"] = D.apply(lambda t: any_match(t, r"DETECTOR|DETECTED BY"))
df["high_temp_mentioned"] = D.apply(lambda t: any_match(t, r"DEGREE|TEMPERATURE|HOT"))
df[["area", "cause_category", "fuel_type", "equipment_type", "operating_activity", "primary_tag"]].head(15)

,area,cause_category,fuel_type,equipment_type,operating_activity,primary_tag
0,Loading Gantry,Vehicle fault,Vehicle materials,Vehicle,Loading / Unloading,NaN
1,RFCC,Pyrophoric,Unknown,Heat exchanger / Heater,Maintenance / Project work,113-EE-1002C
2,MSB,Hot work / Maintenance activity,Unknown,Unknown,Shutdown,NaN
3,MSB,Flange / Seal leak,Hydrocarbon (process),Reactor,Normal operation,152-RB-1001
4,CDU,Hot work / Maintenance activity,Unknown,Piping / Valve / Line,Maintenance / Project work,NaN
5,MHC,Flange / Seal leak,Diesel / Fuel oil,Heat exchanger / Heater,Normal operation,122-EE-1004
6,MSB,Flange / Seal leak,Hydrocarbon (process),Reactor,Normal operation,NaN
7,Offshore,Arson / Vandalism / Scavenging,Electrical,Cable,Normal operation,NaN
8,CDU,Hot work / Maintenance activity,Hydrocarbon (process),Vessel / Drum,Maintenance / Project work,102-VV-1004
9,MHC,Flange / Seal leak,Hydrocarbon (process),Heat exchanger / Heater,Normal operation,121-EE-1004


## 9. Feature extraction from ACTION TAKEN & CLOSURE

In [ ]:
ALL_TXT = (df["emergency_details"].fillna("") + " " + df["action_taken"].fillna("") + " " +
           df["closure_notes"].fillna(""))
A = df["action_taken"].fillna("") + " " + df["closure_notes"].fillna("")

df["agent_dcp"]      = A.apply(lambda t: any_match(t, r"\bDCP\b"))
df["agent_co2"]      = A.apply(lambda t: any_match(t, r"CO2"))
df["agent_water"]    = A.apply(lambda t: any_match(t, r"WATER|HOSE|JET|HYDRANT|RISER|FOG"))
df["agent_foam"]     = A.apply(lambda t: any_match(t, r"FOAM|FFFP"))
df["agent_monitor"]  = A.apply(lambda t: any_match(t, r"MONITOR"))
df["agent_steam_n2"] = A.apply(lambda t: any_match(t, r"STEAM|NITROGEN"))
df["ventilation_used"] = A.apply(lambda t: any_match(t, r"VENTILAT|SMOKE EXTRACT"))
df["isolation_done"] = A.apply(lambda t: any_match(t, r"ISOLAT|SHUT|SHUT THE"))
df["agents_count"] = df[["agent_dcp", "agent_co2", "agent_water", "agent_foam", "agent_steam_n2"]].sum(axis=1)

# number of extinguisher units and total kg, e.g. "2 units of 9kg", "1 x 50kg", "4uints of 9kg", "Two units of 5kg"
WORDNUM = {"ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4, "FIVE": 5, "SIX": 6, "SEVEN": 7, "EIGHT": 8, "NINE": 9, "TEN": 10}
def extinguisher_stats(text):
    t = str(text).upper()
    for w, n in WORDNUM.items():
        t = re.sub(rf"\b{w}\b", str(n), t)
    pair_re = r"(\d+)\s*(?:X|UNITS?|UINTS?)\s*(?:OF)?\s*(\d+)\s*KG"
    pairs = re.findall(pair_re, t)
    units = sum(int(u) for u, _ in pairs)
    kg = sum(int(u) * int(k) for u, k in pairs)
    # remaining single mentions without a count, e.g. "with 9kg DCP" -> 1 unit
    rest = re.sub(pair_re, " ", t)
    singles = re.findall(r"(\d+)\s*KG", rest)
    units += len(singles)
    kg += sum(int(k) for k in singles)
    return units, kg

st = A.apply(extinguisher_stats)
df["extinguisher_units"] = [s[0] for s in st]
df["extinguisher_total_kg"] = [s[1] for s in st]
df["hose_lines"] = A.apply(lambda t: len(re.findall(r"LINE[S]? OF HOSE|HOSE LINE|HOSE REEL|LENGTH OF HOSE|DELIVERY HOSE", str(t).upper())))

df["out_before_fire_crew"] = A.apply(lambda t: any_match(
    t, r"OUT BEFORE|BEFORE THE ARRIVAL|BEFORE ARRIVAL|OUT ON (THE )?ARRIVAL|EXTINGUISHED ITSELF|STOPPED BEFORE"))
df["crew_on_standby_at_scene"] = ALL_TXT.apply(lambda t: any_match(
    t, r"ALREADY ON STANDBY|ON STANDBY AT THE LOCATION|WHILE FIRE CREW WERE ON STANDBY|CREW ON STANDBY|WHILE THE FIRE TEAM WERE ON STANDBY|FOR A FIRE COVER"))

EXT_BY_RULES = [
    (r"SECURITY", "Security"),
    (r"ELECTRICAL (TEAM|OPERATOR|PERSONNEL)", "Electrical team"),
    (r"OPERATION|OPERATOR|FIELD OFFICER|PLANT", "Operations team"),
    (r"CONTRACTOR|ENERGIO", "Contractor"),
    (r"OCCUPANT|CHEF|SITE OFFICALS|SITE OFFICIALS", "Occupants / Others"),
    (r"FERTILIZER", "Third-party fire team"),
    (r"ITSELF", "Self-extinguished"),
]
def extinguished_by(row):
    a = str(row["action_taken"]).upper() if pd.notna(row["action_taken"]) else ""
    if row["out_before_fire_crew"] == 1:
        return first_match(a, EXT_BY_RULES, "Others")
    return "Fire crew"
df["extinguished_by"] = df.apply(extinguished_by, axis=1)

df["injury_reported"] = ALL_TXT.apply(lambda t: any_match(t, r"INJUR|BURN\b|DEGREE BURN|CASUALTY|VICTIM|MEDICAL"))
df["fatality_reported"] = ALL_TXT.apply(lambda t: any_match(t, r"FATAL|DEATH|DIED|DECEASED"))
df["recurrence_risk_noted"] = ALL_TXT.apply(lambda t: any_match(t, r"REOCC?UR|RE-?IGNIT|RECURRENCE|REOCCRANCE|FORESEEABLE"))
df["after_emergency_standby"] = ALL_TXT.apply(lambda t: any_match(t, r"REMAIN(ED)? ON STANDBY|MAINTAINED AN? .*STANDBY|STAYED"))
df["extinguishers_replaced"] = ALL_TXT.apply(lambda t: any_match(t, r"REPLACE"))

df[["action_taken", "extinguisher_units", "extinguisher_total_kg", "hose_lines",
    "out_before_fire_crew", "extinguished_by", "injury_reported"]].head(12)

,action_taken,extinguisher_units,extinguisher_total_kg,hose_lines,out_before_fire_crew,extinguished_by,injury_reported
0,On arrival the fire crew immediately put off the fire wi...,0,0,0,0,Fire crew,1
1,Out on the arrival of fire crew by the plant field offic...,0,0,0,1,Operations team,0
2,Out on the arrival of fire crew by the plant field offic...,0,0,0,1,Operations team,0
3,"On arrival, the fire crew immediately put off the fire w...",1,9,0,0,Fire crew,0
4,During the puncher of line minor spark observed and exti...,0,0,0,1,Self-extinguished,0
5,On arrival the fire crew immediately put off the fire us...,0,0,1,0,Fire crew,0
6,Out on the arrival of fire crew by the plant field offic...,0,0,0,1,Operations team,0
7,Out before the arrival of the fire crew by the site offi...,0,0,0,1,Occupants / Others,0
8,Out before the arrival of the fire crew by Energio contr...,1,6,2,1,Contractor,0
9,On arrival of the crew they immediately deployed two 100...,0,0,0,0,Fire crew,0


## 10. Target labels
* **fire_nature** – same categories as the `SUMMARY OF FIRE` sheet (BUSH/SCRAP, BUILDING, EQUIPMENT, 3RD PARTY)
* **severity_score / severity_level** – rule-based derived label (adjust weights with the HSE team)

In [ ]:
def fire_nature(row):
    det = str(row["emergency_details"]).upper()
    loc = str(row["location_raw"]).upper()
    if re.search(r"VEHICLE|\bCAR\b|TRUCK|TANKER|HILUX|CATERPILLAR", det):
        return "3RD PARTY"
    if row["zone"] in ("Accommodation", "Admin") or re.search(r"CONTROL ROOM|KITCHEN", loc) \
            or re.search(r"KITCHEN|TOILET|APARTMENT|KETTLE", det):
        return "BUILDING"
    if re.search(r"SCAVENGER|DUMPED|MISCREA|SCRAP|BUSH|GRASS|REFUSE", det):
        return "BUSH/SCRAP"
    return "EQUIPMENT"
df["fire_nature"] = df.apply(fire_nature, axis=1)

score = (
    1
    + 3 * df["injury_reported"] + 5 * df["fatality_reported"]
    + 2 * df["is_explosion_blast"] + 2 * df["fire_spread"]
    + 1 * (df["extinguisher_units"] >= 3) + 1 * (df["hose_lines"] >= 1)
    + 1 * df["agent_foam"] + 1 * df["agents_count"].ge(2)
    + 1 * df["hydrogen_involved"]
    - 1 * df["is_minor"] - 1 * df["is_smoke_only"]
    - 1 * ((df["out_before_fire_crew"] == 1) & (df["extinguisher_units"] <= 1))
).clip(lower=0)
df["severity_score"] = score
df["severity_level"] = pd.cut(score, bins=[-1, 1, 3, 100], labels=["Low", "Medium", "High"]).astype(str)

print(df["fire_nature"].value_counts(), "\n")
print(df["severity_level"].value_counts(), "\n")
print(df["cause_category"].value_counts())

fire_nature
EQUIPMENT     44
BUILDING      10
3RD PARTY      5
BUSH/SCRAP     2
Name: count, dtype: int64 

severity_level
Low       35
Medium    17
High       9
Name: count, dtype: int64 

cause_category
Flange / Seal leak                 15
Electrical fault                   10
Other / Unknown                    10
Hot work / Maintenance activity     6
Vehicle fault                       5
Cooking / Domestic appliance        3
Hot surface / Soaked insulation     3
Arson / Vandalism / Scavenging      2
Lightning strike                    2
Pyrophoric                          1
Dumped material                     1
Overheating                         1
Equipment failure / Blast           1
Vent release                        1
Name: count, dtype: int64


## 11. Recurrence / history features
For each incident – how often has this **area** or **equipment** had fires before? (Only past data is used – no leakage.)

In [ ]:
df = df.sort_values(["incident_date", "call_datetime"], na_position="last").reset_index(drop=True)
df.insert(0, "incident_id", [f"FIRE-{YEAR}-{i:03d}" for i in range(1, len(df) + 1)])

df["prior_incidents_area"] = df.groupby("area").cumcount()
df["days_since_last_area_fire"] = df.groupby("area")["incident_date"].diff().dt.days
df["days_since_last_plant_fire"] = df["incident_date"].diff().dt.days

df["prior_incidents_same_tag"] = np.where(
    df["primary_tag_base"].notna(), df.groupby("primary_tag_base").cumcount(), 0)
df["is_repeat_equipment"] = (df["prior_incidents_same_tag"] > 0).astype(int)

def rolling_prior(group_col, days):
    out = []
    for _, r in df.iterrows():
        m = (df[group_col] == r[group_col]) & (df["incident_date"] < r["incident_date"]) & \
            (df["incident_date"] >= r["incident_date"] - pd.Timedelta(days=days))
        out.append(int(m.sum()))
    return out

df["area_fires_prev_30d"] = rolling_prior("area", 30)
df["area_fires_prev_90d"] = rolling_prior("area", 90)
df["_all"] = 1
df["plant_fires_prev_7d"] = rolling_prior("_all", 7)
df["plant_fires_prev_30d"] = rolling_prior("_all", 30)
df = df.drop(columns="_all")

df.groupby("primary_tag_base").size().sort_values(ascending=False).head(10).rename("fires_per_equipment_tag")

primary_tag_base
121-EE-1004    8
121-EE-1005    2
716-TT-1601    2
112-PA-1006    1
102-VV-1004    1
111-PA-1001    1
112-EE-1008    1
113-EE-1021    1
113-EE-1002    1
112-PA-1014    1
Name: fires_per_equipment_tag, dtype: int64

## 12. Final clean incident table

In [ ]:
incident_cols = [
    "incident_id", "incident_date", "call_datetime", "response_datetime", "response_minutes",
    "month", "month_name", "quarter", "week_of_year", "day_of_month", "day_of_week", "day_name",
    "is_weekend", "hour", "shift", "time_band", "season", "hour_sin", "hour_cos", "month_sin", "month_cos",
    "location_clean", "area", "zone", "unit_number", "equipment_tags", "primary_tag", "primary_tag_base",
    "equipment_type", "cause_category", "fuel_type", "operating_activity",
    "is_flash_fire", "is_smoke_only", "is_explosion_blast", "is_minor", "fire_spread",
    "hydrocarbon_involved", "hydrogen_involved", "contractor_involved", "detected_by_system", "high_temp_mentioned",
    "agent_dcp", "agent_co2", "agent_water", "agent_foam", "agent_monitor", "agent_steam_n2",
    "ventilation_used", "isolation_done", "agents_count", "extinguisher_units", "extinguisher_total_kg", "hose_lines",
    "out_before_fire_crew", "crew_on_standby_at_scene", "extinguished_by",
    "injury_reported", "fatality_reported", "recurrence_risk_noted", "after_emergency_standby", "extinguishers_replaced",
    "prior_incidents_area", "days_since_last_area_fire", "days_since_last_plant_fire",
    "prior_incidents_same_tag", "is_repeat_equipment",
    "area_fires_prev_30d", "area_fires_prev_90d", "plant_fires_prev_7d", "plant_fires_prev_30d",
    "fire_nature", "severity_score", "severity_level",
    "time_missing_flag", "date_fix_flag",
    "emergency_details", "action_taken", "closure_notes",
    "SHEET_MONTH", "SOURCE_ROW", "date_raw", "call_time_raw", "response_time_raw", "location_raw",
]
incidents = df[incident_cols].copy()
incidents["date_raw"] = incidents["date_raw"].astype(str)
print(incidents.shape)
incidents.head()

(61, 85)


,incident_id,incident_date,call_datetime,response_datetime,response_minutes,month,month_name,quarter,week_of_year,day_of_month,day_of_week,day_name,is_weekend,hour,shift,time_band,season,hour_sin,hour_cos,month_sin,month_cos,location_clean,area,zone,unit_number,equipment_tags,primary_tag,primary_tag_base,equipment_type,cause_category,fuel_type,operating_activity,is_flash_fire,is_smoke_only,is_explosion_blast,is_minor,fire_spread,hydrocarbon_involved,hydrogen_involved,contractor_involved,detected_by_system,high_temp_mentioned,agent_dcp,agent_co2,agent_water,agent_foam,agent_monitor,agent_steam_n2,ventilation_used,isolation_done,agents_count,extinguisher_units,extinguisher_total_kg,hose_lines,out_before_fire_crew,crew_on_standby_at_scene,extinguished_by,injury_reported,fatality_reported,recurrence_risk_noted,after_emergency_standby,extinguishers_replaced,prior_incidents_area,days_since_last_area_fire,days_since_last_plant_fire,prior_incidents_same_tag,is_repeat_equipment,area_fires_prev_30d,area_fires_prev_90d,plant_fires_prev_7d,plant_fires_prev_30d,fire_nature,severity_score,severity_level,time_missing_flag,date_fix_flag,emergency_details,action_taken,closure_notes,SHEET_MONTH,SOURCE_ROW,date_raw,call_time_raw,response_time_raw,location_raw
0,FIRE-2026-001,2026-01-01,2026-01-01 21:16:00,2026-01-01 21:17:00,1.0,1,January,1,1,1,3,Thursday,0,21.0,Night (19-07),Evening (18-24),Dry,-0.707107,7.071068e-01,0.5,0.866025,GANTRY 3 BAY 21,Loading Gantry,Loading Gantry,NaN,NaN,NaN,NaN,Vehicle,Vehicle fault,Vehicle materials,Loading / Unloading,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,Fire crew,1,0,0,0,0,0,NaN,NaN,0.0,0,0,0,0,0,3RD PARTY,5,High,0,ok,Fire involving the loading arm of Bay 21 at Gantry 3 whi...,On arrival the fire crew immediately put off the fire wi...,"After fire was put out, the Tanker driver had a 2nd-Degr...",JANUARY,4,2026-01-01 00:00:00,2116HRS,2117HRS,GANTRY 3 BAY 21
1,FIRE-2026-002,2026-01-06,2026-01-06 03:20:00,2026-01-06 03:22:00,2.0,1,January,1,2,6,1,Tuesday,0,3.0,Night (19-07),Early Morning (00-06),Dry,0.707107,7.071068e-01,0.5,0.866025,RFCC,RFCC,Process Unit,113,"113-EE-1002C, 113-EE-1002",113-EE-1002C,113-EE-1002,Heat exchanger / Heater,Pyrophoric,Unknown,Maintenance / Project work,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,Operations team,0,0,1,0,0,0,NaN,5.0,0.0,0,0,0,1,1,EQUIPMENT,0,Low,0,day_month_swapped,Flash fire reported at 3:20am by KOINS team removing flo...,Out on the arrival of fire crew by the plant field offic...,After the water flushing was done on the exchanger float...,JANUARY,5,2026-06-01 00:00:00,0320HRS,0322HRS,RFCC
2,FIRE-2026-003,2026-01-13,2026-01-13 14:00:00,2026-01-13 14:02:00,2.0,1,January,1,3,13,1,Tuesday,0,14.0,Day (07-19),Afternoon (12-18),Dry,-0.500000,-8.660254e-01,0.5,0.866025,MSB BATTERY LIMIT,MSB,Process Unit,NaN,NaN,NaN,NaN,Unknown,Hot work / Maintenance activity,Unknown,Shutdown,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,0,0,1,0,Operations team,0,0,0,1,0,0,NaN,7.0,0.0,0,0,0,1,2,EQUIPMENT,0,Low,0,ok,Fire during hot job at plant battery limit. Job was MOC ...,Out on the arrival of fire crew by the plant field offic...,"After the hot work activities, the fire crew return to t...",JANUARY,6,13/1/2026,1400HRS,1402HRS,MSB Battery limit
3,FIRE-2026-004,2026-01-22,2026-01-22 18:39:00,2026-01-22 18:40:00,1.0,1,January,1,4,22,3,Thursday,0,18.0,Day (07-19),Evening (18-24),Dry,-1.000000,-1.836970e-16,0.5,0.866025,MSB CCR UNIT 153,MSB,Process Unit,152,152-RB-1001,152-RB-1001,152-RB-1001,Reactor,Flange / Seal leak,Hydrocarbon (process),Normal operation,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,9,0,0,0,Fire crew,0,0,0,1,0,1,9.0,9.0,0.0,0,1,1,0,3,EQUIPMENT,1,Low,0,ok,Fire confined to the flanges of a reactor 152-RB-1001 re...,"On arrival, the fire crew immediately put off the fire w...","After the fire was put off, SFS-2 crew remain on standby...",JANUARY,7,22/1/2026,1839HRS,1840HRS,MSB CCR UNIT 153
4,FIRE-2026-005,2026-01-24,2026-01-24 22:40:00,2026-01-24 22:41:00,1.0,1,January,1,4,24

## 13. Data quality report

In [ ]:
dq = pd.DataFrame({
    "dtype": incidents.dtypes.astype(str),
    "missing": incidents.isna().sum(),
    "missing_%": (incidents.isna().mean() * 100).round(1),
    "unique": incidents.nunique(),
})
issues = pd.DataFrame([
    ["Dates stored as MM/DD and swapped", int((df["date_fix_flag"] == "day_month_swapped").sum())],
    ["Missing call time", int(df["call_time"].isna().sum())],
    ["Missing response time", int(df["response_time"].isna().sum())],
    ["Missing action taken", int(df["action_taken"].isna().sum())],
    ["Missing closure notes", int(df["closure_notes"].isna().sum())],
    ["Area mapped to 'Other'", int((df["area"] == "Other").sum())],
    ["Cause = Other / Unknown", int((df["cause_category"] == "Other / Unknown").sum())],
    ["Negative / >120 min response", int(((df["response_minutes"] < 0) | (df["response_minutes"] > 120)).sum())],
], columns=["check", "count"])
display(issues) if "display" in dir() else print(issues)
dq[dq["missing"] > 0]

                               check  count
0  Dates stored as MM/DD and swapped      1
1                  Missing call time      1
2              Missing response time      1
3               Missing action taken      1
4              Missing closure notes      1
5             Area mapped to 'Other'      0
6            Cause = Other / Unknown     10
7       Negative / >120 min response      0


,dtype,missing,missing_%,unique
call_datetime,datetime64[us],1,1.6,60
response_datetime,datetime64[us],1,1.6,60
response_minutes,float64,1,1.6,3
hour,float64,1,1.6,23
hour_sin,float64,1,1.6,20
hour_cos,float64,1,1.6,22
unit_number,object,29,47.5,15
equipment_tags,str,29,47.5,25
primary_tag,object,29,47.5,24
primary_tag_base,object,29,47.5,23


## 14. Validate against the SUMMARY OF FIRE sheet

In [ ]:
summ_raw = pd.read_excel(INPUT_FILE, sheet_name=[s for s in xls.sheet_names if "SUMMARY" in s.upper()][0], header=None)
start = summ_raw.index[summ_raw.iloc[:, 1].astype(str).str.strip().str.upper().isin(MONTHS.keys())]
summ = summ_raw.loc[start, [1, 2, 4, 5, 6, 7]].copy()
summ.columns = ["MONTH", "SUMMARY_TOTAL", "SUMMARY_BUSH/SCRAP", "SUMMARY_BUILDING", "SUMMARY_EQUIPMENT", "SUMMARY_3RD PARTY"]
summ["MONTH"] = summ["MONTH"].str.strip().str.upper()
for c in summ.columns[1:]:
    summ[c] = pd.to_numeric(summ[c].replace(["-", "\\-"], 0), errors="coerce").fillna(0).astype(int)

calc = (incidents.assign(MONTH=incidents["month_name"].str.upper())
        .pivot_table(index="MONTH", columns="fire_nature", values="incident_id", aggfunc="count", fill_value=0))
calc = calc.reindex(columns=["BUSH/SCRAP", "BUILDING", "EQUIPMENT", "3RD PARTY"], fill_value=0).add_prefix("CALC_")
calc["CALC_TOTAL"] = calc.sum(axis=1)

validation = summ.merge(calc.reset_index(), on="MONTH", how="outer").fillna(0)
validation["MONTH_NUM"] = validation["MONTH"].map(MONTHS)
validation = validation.sort_values("MONTH_NUM").drop(columns="MONTH_NUM")
validation["TOTAL_MATCH"] = np.where(validation["SUMMARY_TOTAL"] == validation["CALC_TOTAL"], "YES", "NO")
for cat in ["BUSH/SCRAP", "BUILDING", "EQUIPMENT", "3RD PARTY"]:
    validation[f"DIFF_{cat}"] = validation[f"CALC_{cat}"] - validation[f"SUMMARY_{cat}"]
print("Summary total:", int(validation["SUMMARY_TOTAL"].sum()), "| Records extracted:", len(incidents))
validation

Summary total: 61 | Records extracted: 61


,MONTH,SUMMARY_TOTAL,SUMMARY_BUSH/SCRAP,SUMMARY_BUILDING,SUMMARY_EQUIPMENT,SUMMARY_3RD PARTY,CALC_BUSH/SCRAP,CALC_BUILDING,CALC_EQUIPMENT,CALC_3RD PARTY,CALC_TOTAL,TOTAL_MATCH,DIFF_BUSH/SCRAP,DIFF_BUILDING,DIFF_EQUIPMENT,DIFF_3RD PARTY
3,JANUARY,7,0,0,6,1,0,0,6,1,7,YES,0,0,0,0
2,FEBRUARY,11,2,2,7,0,2,2,7,0,11,YES,0,0,0,0
6,MARCH,2,0,0,1,1,0,0,1,1,2,YES,0,0,0,0
0,APRIL,5,0,0,5,0,0,0,5,0,5,YES,0,0,0,0
7,MAY,4,0,0,4,0,0,0,4,0,4,YES,0,0,0,0
5,JUNE,12,0,5,7,0,0,4,8,0,12,YES,0,-1,1,0
4,JULY,11,0,3,8,0,0,3,8,0,11,YES,0,0,0,0
1,AUGUST,5,0,0,3,2,0,0,3,2,5,YES,0,0,0,0
9,SEPTEMBER,3,0,0,2,1,0,1,2,0,3,YES,0,1,0,-1
8,OCTOBER,1,0,0,0,1,0,0,0,1,1,YES,0,0,0,0


## 15. ML dataset A – Incident level (encoded)
Use for **classification**: predict `severity_level`, `fire_nature`, `cause_category` or `out_before_fire_crew`.
Categorical columns are one-hot encoded; text/ID columns removed.

In [ ]:
TARGETS = ["fire_nature", "severity_level", "severity_score", "cause_category"]
CAT_FEATURES = ["shift", "time_band", "season", "day_name", "area", "zone", "equipment_type",
                "fuel_type", "operating_activity", "extinguished_by"]
NUM_FEATURES = ["month", "quarter", "week_of_year", "day_of_week", "is_weekend", "hour",
                "hour_sin", "hour_cos", "month_sin", "month_cos", "response_minutes",
                "is_flash_fire", "is_smoke_only", "is_explosion_blast", "is_minor", "fire_spread",
                "hydrocarbon_involved", "hydrogen_involved", "contractor_involved", "detected_by_system",
                "high_temp_mentioned", "agent_dcp", "agent_co2", "agent_water", "agent_foam", "agent_monitor",
                "agent_steam_n2", "ventilation_used", "isolation_done", "agents_count", "extinguisher_units",
                "extinguisher_total_kg", "hose_lines", "out_before_fire_crew", "crew_on_standby_at_scene",
                "injury_reported", "recurrence_risk_noted", "prior_incidents_area", "days_since_last_area_fire",
                "days_since_last_plant_fire", "prior_incidents_same_tag", "is_repeat_equipment",
                "area_fires_prev_30d", "area_fires_prev_90d", "plant_fires_prev_7d", "plant_fires_prev_30d"]

ml_incident = incidents[["incident_id", "incident_date"] + NUM_FEATURES + CAT_FEATURES + TARGETS].copy()
# fill numerics: unknown hour -> median, first-occurrence gaps -> -1 (= no previous fire)
ml_incident["hour"] = ml_incident["hour"].fillna(ml_incident["hour"].median())
for c in ["hour_sin", "hour_cos"]:
    ml_incident[c] = ml_incident[c].fillna(0)
ml_incident["response_minutes"] = ml_incident["response_minutes"].fillna(ml_incident["response_minutes"].median())
for c in ["days_since_last_area_fire", "days_since_last_plant_fire"]:
    ml_incident[c] = ml_incident[c].fillna(-1)

ml_incident_encoded = pd.get_dummies(ml_incident, columns=CAT_FEATURES, prefix=CAT_FEATURES, dtype=int)
print(ml_incident_encoded.shape)
ml_incident_encoded.head()

(61, 138)


,incident_id,incident_date,month,quarter,week_of_year,day_of_week,is_weekend,hour,hour_sin,hour_cos,month_sin,month_cos,response_minutes,is_flash_fire,is_smoke_only,is_explosion_blast,is_minor,fire_spread,hydrocarbon_involved,hydrogen_involved,contractor_involved,detected_by_system,high_temp_mentioned,agent_dcp,agent_co2,agent_water,agent_foam,agent_monitor,agent_steam_n2,ventilation_used,isolation_done,agents_count,extinguisher_units,extinguisher_total_kg,hose_lines,out_before_fire_crew,crew_on_standby_at_scene,injury_reported,recurrence_risk_noted,prior_incidents_area,days_since_last_area_fire,days_since_last_plant_fire,prior_incidents_same_tag,is_repeat_equipment,area_fires_prev_30d,area_fires_prev_90d,plant_fires_prev_7d,plant_fires_prev_30d,fire_nature,severity_level,...,area_Surge Tank,zone_Accommodation,zone_Admin,zone_Gate & Roads,zone_Loading Gantry,zone_Offshore,zone_Process Unit,zone_Tank Farm,zone_Third-Party Premises,zone_Utilities & Power,equipment_type_Blower / Fan,equipment_type_Building structure,equipment_type_Cable,equipment_type_Compressor,equipment_type_Domestic appliance,equipment_type_Electrical panel,equipment_type_Heat exchanger / Heater,equipment_type_Piping / Valve / Line,equipment_type_Pump,equipment_type_Reactor,equipment_type_Storage tank,equipment_type_Turbine / Expander,equipment_type_Unknown,equipment_type_Vehicle,equipment_type_Vessel / Drum,fuel_type_Diesel / Fuel oil,fuel_type_Domestic / Cooking,fuel_type_Electrical,fuel_type_Gasoline / Naphtha,fuel_type_Hydrocarbon (process),fuel_type_Hydrogen,fuel_type_Insulation material,fuel_type_Natural gas,fuel_type_Unknown,fuel_type_Vehicle materials,operating_activity_Loading / Unloading,operating_activity_Maintenance / Project work,operating_activity_Normal operation,operating_activity_Process change,operating_activity_Shutdown,operating_activity_Start-up,extinguished_by_Contractor,extinguished_by_Electrical team,extinguished_by_Fire crew,extinguished_by_Occupants / Others,extinguished_by_Operations team,extinguished_by_Others,extinguished_by_Security,extinguished_by_Self-extinguished,extinguished_by_Third-party fire team
0,FIRE-2026-001,2026-01-01,1,1,1,3,0,21.0,-0.707107,7.071068e-01,0.5,0.866025,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,-1.0,-1.0,0.0,0,0,0,0,0,3RD PARTY,High,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,FIRE-2026-002,2026-01-06,1,1,2,1,0,3.0,0.707107,7.071068e-01,0.5,0.866025,2.0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,1,0,-1.0,5.0,0.0,0,0,0,1,1,EQUIPMENT,Low,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,FIRE-2026-003,2026-01-13,1,1,3,1,0,14.0,-0.500000,-8.660254e-01,0.5,0.866025,2.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,-1.0,7.0,0.0,0,0,0,1,2,EQUIPMENT,Low,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
3,FIRE-2026-004,2026-01-22,1,1,4,3,0,18.0,-1.000000,-1.836970e-16,0.5,0.866025,1.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,9,0,0,0,0,0,1,9.0,9.0,0.0,0,1,1,0,3,EQUIPMENT,Low,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
4,FIRE-2026-005,2026-01-24,1,1,4,5,1,22.0,-0.500000,8.660254e-01,0.5,0.866025,1.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,2.0,0.0,0,0,0,1,4,EQUIPMENT,Low,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0


## 16. ML dataset B – Daily × Area panel (fire occurrence prediction)
One row per **day per area** with `fire_occurred` (0/1) as target, plus lag features computed only from **previous** days.
This is the main dataset for *"predict where and when the next fire is likely"*.

In [ ]:
start_date = pd.Timestamp(YEAR, 1, 1)
end_date = (incidents["incident_date"].max() + pd.offsets.MonthEnd(0))
all_days = pd.date_range(start_date, end_date, freq="D")
areas = sorted(incidents["area"].unique())

panel = pd.MultiIndex.from_product([all_days, areas], names=["date", "area"]).to_frame(index=False)
daily_counts = incidents.groupby(["incident_date", "area"]).size().rename("fire_count").reset_index() \
                        .rename(columns={"incident_date": "date"})
panel = panel.merge(daily_counts, on=["date", "area"], how="left").fillna({"fire_count": 0})
panel["fire_count"] = panel["fire_count"].astype(int)
panel["fire_occurred"] = (panel["fire_count"] > 0).astype(int)
panel["zone"] = panel["area"].map(incidents.drop_duplicates("area").set_index("area")["zone"])

panel = panel.sort_values(["area", "date"]).reset_index(drop=True)
g = panel.groupby("area")["fire_count"]
for w in [7, 14, 30, 60, 90]:
    panel[f"area_fires_prev_{w}d"] = g.transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum()).fillna(0)
panel["area_cum_fires_before"] = g.transform(lambda s: s.shift(1).cumsum()).fillna(0)

def days_since_last(s):
    out, last = [], None
    for i, v in enumerate(s.values):
        out.append(np.nan if last is None else i - last)
        if v > 0: last = i
    return pd.Series(out, index=s.index)
panel["days_since_last_area_fire"] = g.transform(days_since_last).fillna(999)

plant_daily = panel.groupby("date")["fire_count"].sum()
plant_prev7 = plant_daily.shift(1).rolling(7, min_periods=1).sum().fillna(0)
plant_prev30 = plant_daily.shift(1).rolling(30, min_periods=1).sum().fillna(0)
panel["plant_fires_prev_7d"] = panel["date"].map(plant_prev7)
panel["plant_fires_prev_30d"] = panel["date"].map(plant_prev30)

panel["month"] = panel["date"].dt.month
panel["day_of_week"] = panel["date"].dt.dayofweek
panel["is_weekend"] = (panel["day_of_week"] >= 5).astype(int)
panel["week_of_year"] = panel["date"].dt.isocalendar().week.astype(int)
panel["season"] = np.where(panel["month"].between(4, 10), "Rainy", "Dry")
panel["month_sin"] = np.sin(2 * np.pi * panel["month"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["month"] / 12)
panel["dow_sin"] = np.sin(2 * np.pi * panel["day_of_week"] / 7)
panel["dow_cos"] = np.cos(2 * np.pi * panel["day_of_week"] / 7)

panel = panel.sort_values(["date", "area"]).reset_index(drop=True)
print(panel.shape, "| positive rate:", round(panel["fire_occurred"].mean() * 100, 2), "%")
panel_encoded = pd.get_dummies(panel, columns=["area", "zone", "season"], dtype=int)
panel.head()

(6080, 23) | positive rate: 0.95 %


,date,area,fire_count,fire_occurred,zone,area_fires_prev_7d,area_fires_prev_14d,area_fires_prev_30d,area_fires_prev_60d,area_fires_prev_90d,area_cum_fires_before,days_since_last_area_fire,plant_fires_prev_7d,plant_fires_prev_30d,month,day_of_week,is_weekend,week_of_year,season,month_sin,month_cos,dow_sin,dow_cos
0,2026-01-01,10K Accommodation,0,0,Accommodation,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0,0.0,1,3,0,1,Dry,0.5,0.866025,0.433884,-0.900969
1,2026-01-01,22K Accommodation,0,0,Accommodation,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0,0.0,1,3,0,1,Dry,0.5,0.866025,0.433884,-0.900969
2,2026-01-01,Admin Block,0,0,Admin,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0,0.0,1,3,0,1,Dry,0.5,0.866025,0.433884,-0.900969
3,2026-01-01,CDU,0,0,Process Unit,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0,0.0,1,3,0,1,Dry,0.5,0.866025,0.433884,-0.900969
4,2026-01-01,CNG Dispenser,0,0,Loading Gantry,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0,0.0,1,3,0,1,Dry,0.5,0.866025,0.433884,-0.900969


## 17. ML dataset C – Daily plant-level time series

In [ ]:
daily = pd.DataFrame({"date": all_days})
cnt = incidents.groupby("incident_date").agg(
    fire_count=("incident_id", "count"),
    equipment_fires=("fire_nature", lambda s: (s == "EQUIPMENT").sum()),
    building_fires=("fire_nature", lambda s: (s == "BUILDING").sum()),
    third_party_fires=("fire_nature", lambda s: (s == "3RD PARTY").sum()),
    bush_scrap_fires=("fire_nature", lambda s: (s == "BUSH/SCRAP").sum()),
    high_severity=("severity_level", lambda s: (s == "High").sum()),
    injuries=("injury_reported", "sum"),
).reset_index().rename(columns={"incident_date": "date"})
daily = daily.merge(cnt, on="date", how="left").fillna(0)
daily["fire_occurred"] = (daily["fire_count"] > 0).astype(int)
for w in [1, 7, 14, 30]:
    daily[f"fires_prev_{w}d"] = daily["fire_count"].shift(1).rolling(w, min_periods=1).sum().fillna(0)
daily["fires_next_7d"] = daily["fire_count"][::-1].shift(1).rolling(7, min_periods=1).sum()[::-1].fillna(0)  # forecast target
daily["month"] = daily["date"].dt.month
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)
daily["season"] = np.where(daily["month"].between(4, 10), "Rainy", "Dry")
print(daily.shape)
daily[daily["fire_count"] > 0].head()

(304, 18)


,date,fire_count,equipment_fires,building_fires,third_party_fires,bush_scrap_fires,high_severity,injuries,fire_occurred,fires_prev_1d,fires_prev_7d,fires_prev_14d,fires_prev_30d,fires_next_7d,month,day_of_week,is_weekend,season
0,2026-01-01,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1,0.0,0.0,0.0,0.0,1.0,1,3,0,Dry
5,2026-01-06,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0.0,1.0,1.0,1.0,1.0,1,1,0,Dry
12,2026-01-13,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0.0,1.0,2.0,2.0,0.0,1,1,0,Dry
21,2026-01-22,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,1.0,3.0,2.0,1,3,0,Dry
23,2026-01-24,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0.0,1.0,2.0,4.0,2.0,1,5,1,Dry


## 18. Quick EDA summaries

In [ ]:
eda = {
    "by_area": incidents["area"].value_counts().rename_axis("area").reset_index(name="fires"),
    "by_cause": incidents["cause_category"].value_counts().rename_axis("cause").reset_index(name="fires"),
    "by_equipment": incidents["equipment_type"].value_counts().rename_axis("equipment_type").reset_index(name="fires"),
    "by_time_band": incidents["time_band"].value_counts().rename_axis("time_band").reset_index(name="fires"),
    "top_repeat_tags": incidents.dropna(subset=["primary_tag_base"]).groupby("primary_tag_base")
                         .agg(fires=("incident_id", "count"), area=("area", "first"),
                              first_fire=("incident_date", "min"), last_fire=("incident_date", "max"))
                         .sort_values("fires", ascending=False).reset_index(),
}
pivot_area_month = incidents.pivot_table(index="area", columns="month", values="incident_id",
                                         aggfunc="count", fill_value=0)
for k, v in eda.items():
    print(f"\n--- {k} ---"); print(v.head(10).to_string(index=False))
pivot_area_month


--- by_area ---
             area  fires
              MHC     13
             RFCC      9
22K Accommodation      7
              MSB      6
  ITF (Tank Farm)      4
              CDU      3
   Loading Gantry      2
         Offshore      2
              HGU      2
       Substation      2

--- by_cause ---
                          cause  fires
             Flange / Seal leak     15
               Electrical fault     10
                Other / Unknown     10
Hot work / Maintenance activity      6
                  Vehicle fault      5
   Cooking / Domestic appliance      3
Hot surface / Soaked insulation      3
 Arson / Vandalism / Scavenging      2
               Lightning strike      2
                     Pyrophoric      1

--- by_equipment ---
         equipment_type  fires
Heat exchanger / Heater     19
                   Pump      6
                Vehicle      5
                Unknown      4
                  Cable      4
           Storage tank      4
  Piping / Valve / Lin

month,1,2,3,4,5,6,7,8,9,10
area,,,,,,,,,,
10K Accommodation,0,0,0,0,0,0,0,0,1,0
22K Accommodation,0,0,0,0,0,4,3,0,0,0
Admin Block,0,1,0,0,0,0,0,0,0,0
CDU,1,2,0,0,0,0,0,0,0,0
CNG Dispenser,0,0,0,0,0,0,1,0,0,0
CPP,0,0,0,0,1,0,0,0,0,0
Control Room,0,1,0,0,0,0,0,0,0,0
Cooling Tower,0,0,0,0,0,0,0,1,0,0
Fertilizer Premises,0,0,1,0,0,0,0,0,0,0


## 19. Data dictionary

In [ ]:
DESCRIPTIONS = {
    "incident_id": "Unique incident ID", "incident_date": "Cleaned incident date (sheet month validated)",
    "call_datetime": "Date + time emergency call received", "response_datetime": "Date + time fire crew responded",
    "response_minutes": "Response time in minutes (response − call)", "shift": "Day 07-19 / Night 19-07",
    "time_band": "6-hour band of call", "season": "Rainy (Apr–Oct) / Dry (Nov–Mar)",
    "area": "Standardised plant area / unit", "zone": "Higher-level zone grouping",
    "unit_number": "3-digit unit number from equipment tag", "equipment_tags": "All equipment tags found in details",
    "primary_tag": "First equipment tag", "primary_tag_base": "Tag without train suffix (A/B/C)",
    "equipment_type": "Equipment family involved", "cause_category": "Ignition cause (keyword rules)",
    "fuel_type": "Material that burned", "operating_activity": "Plant activity at time of fire",
    "extinguisher_units": "No. of portable extinguishers used", "extinguisher_total_kg": "Total kg of extinguishant used",
    "hose_lines": "No. of hose lines / reels mentioned", "out_before_fire_crew": "1 = fire out before fire crew arrived",
    "extinguished_by": "Who put the fire out", "injury_reported": "1 = injury / casualty mentioned",
    "prior_incidents_area": "Fires at same area before this one (2026)",
    "prior_incidents_same_tag": "Fires on same equipment before this one",
    "is_repeat_equipment": "1 = equipment had a previous fire",
    "fire_nature": "BUSH/SCRAP, BUILDING, EQUIPMENT, 3RD PARTY (summary-sheet categories)",
    "severity_score": "Derived severity points", "severity_level": "Low / Medium / High from severity_score",
    "date_fix_flag": "ok / day_month_swapped / month_mismatch_check", "SOURCE_ROW": "Row number in source sheet",
}
def describe(c):
    if c in DESCRIPTIONS: return DESCRIPTIONS[c]
    if c.startswith("agent_"): return f"1 = {c[6:].upper().replace('_', '/')} used"
    if c.startswith("is_") or c.endswith("_involved"): return "Binary flag (1 = yes) from text"
    if "prev_" in c: return "Count of fires in previous N days (no leakage)"
    if "days_since" in c: return "Days since previous fire"
    if c.endswith(("_sin", "_cos")): return "Cyclical encoding"
    return ""
data_dict = pd.DataFrame({"column": incidents.columns,
                          "dtype": incidents.dtypes.astype(str).values,
                          "description": [describe(c) for c in incidents.columns]})
data_dict

,column,dtype,description
0,incident_id,str,Unique incident ID
1,incident_date,datetime64[us],Cleaned incident date (sheet month validated)
2,call_datetime,datetime64[us],Date + time emergency call received
3,response_datetime,datetime64[us],Date + time fire crew responded
4,response_minutes,float64,Response time in minutes (response − call)
...,...,...,...
80,SOURCE_ROW,int64,Row number in source sheet
81,date_raw,str,
82,call_time_raw,str,
83,response_time_raw,str,


## 20. EXPORT
Writes one formatted Excel workbook (all datasets) + individual CSV files for direct use in ML pipelines.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def export_all(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    xlsx_path = output_dir / f"Fire_Incidents_ML_Dataset_{stamp}.xlsx"

    sheets = {
        "Incidents_Clean": incidents,
        "ML_Incident_Encoded": ml_incident_encoded,
        "ML_Daily_Area_Panel": panel,
        "ML_Daily_Plant": daily,
        "Validation_vs_Summary": validation,
        "Repeat_Equipment": eda["top_repeat_tags"],
        "Area_by_Month": pivot_area_month.reset_index(),
        "Data_Quality": issues,
        "Data_Dictionary": data_dict,
    }

    # Excel can't store tz/time objects nicely – convert
    def prep(d):
        d = d.copy()
        for c in d.columns:
            if d[c].dtype == object:
                d[c] = d[c].apply(lambda v: v.strftime("%H:%M") if isinstance(v, time) else v)
        return d

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
        for name, d in sheets.items():
            prep(d).to_excel(xw, sheet_name=name[:31], index=False)

    # formatting: bold header, fill, freeze panes, filter, column widths, Arial font
    wb = load_workbook(xlsx_path)
    hdr_fill = PatternFill("solid", start_color="1F4E78")
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font = Font(name="Arial", bold=True, color="FFFFFF")
            cell.fill = hdr_fill
            cell.alignment = Alignment(wrap_text=True, vertical="center")
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.font = Font(name="Arial", size=10)
                if cell.is_date:
                    cell.number_format = "dd-mmm-yyyy hh:mm" if "datetime" in str(ws.cell(1, cell.column).value) else "dd-mmm-yyyy"
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions
        for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), start=1):
            width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[get_column_letter(i)].width = min(max(10, width + 2), 60)
    wb.save(xlsx_path)

    # CSVs (UTF-8 with BOM so Excel opens them cleanly)
    csv_paths = []
    for name in ["Incidents_Clean", "ML_Incident_Encoded", "ML_Daily_Area_Panel", "ML_Daily_Plant"]:
        p = output_dir / f"{name.lower()}.csv"
        prep(sheets[name]).to_csv(p, index=False, encoding="utf-8-sig")
        csv_paths.append(p)

    print("Exported:")
    print("  ", xlsx_path.resolve())
    for p in csv_paths:
        print("  ", p.resolve())
    return xlsx_path, csv_paths

xlsx_file, csv_files = export_all()

Exported:
   C:\Users\Work\Downloads\DDF_Project\fire_ml_output\Fire_Incidents_ML_Dataset_20261104_0807.xlsx
   C:\Users\Work\Downloads\DDF_Project\fire_ml_output\incidents_clean.csv
   C:\Users\Work\Downloads\DDF_Project\fire_ml_output\ml_incident_encoded.csv
   C:\Users\Work\Downloads\DDF_Project\fire_ml_output\ml_daily_area_panel.csv
   C:\Users\Work\Downloads\DDF_Project\fire_ml_output\ml_daily_plant.csv
